# 10. Diagrams meet CAD: M1 and M0

Tutorial 6's structure diagram and tutorial 7's 3D viewer, wired so a
click in either one lights up the other. The wiring is fun; the point
is what the round trip *teaches*: the two panes live on different
levels of the modeling stack. The diagram draws **M1**, the model --
one `motors : Motor [4]` node, a description with a multiplicity. The
3D scene renders **M0**, an interpretation -- four actual motor
meshes, one per individual. Cross-selection makes the level jump
visible in both directions.

You will learn how to:

- read `motors : Motor [4]` as *one* M1 element, not four;
- build the M0 population with `m0.interpret` and roll up mass over
  the individuals that exist (tutorial 9's machinery);
- bake per-instance geometry (`drone_geometry(split_instances=True)`)
  and key each mesh part to an M0 individual id;
- watch the M1 -> M0 **fan-out**: selecting the one usage highlights
  all four instance meshes;
- watch the M0 -> M1 **projection**: picking one motor selects the one
  diagram node, while `on_pick` reports *which* individual was hit;
- run the **geometric requirement checks** (camera view-cone
  occlusion, propeller-disc overlap) against the same parametric
  geometry, score them on the requirements scoreboard, and paint the
  offending parts in 3D;
- replay the model's flight state machine over a real globe.

Prerequisites: tutorial 6 (diagrams, `on_select`) and tutorial 9
(`longeron.m0`). The widgets need the `viz` extra and JupyterLab for
pixels, but every cell below also runs headless -- browser clicks are
simulated by writing the same traitlets the front-ends write, and
assertions after each step prove the claims.

In [ ]:
import json

import ipywidgets as W

import longeron
from longeron import diagrams, m0
from longeron.analysis import geometry, link, viewer3d

model = longeron.load("../examples/drone.sysml")

## M1: one usage, whatever the multiplicity says

`Drone::QuadCopter` owns `part motors : Motor[4]` and
`part propellers : Propeller[4]`. At M1 each of those is a *single*
element -- a description saying "four of these exist", not four of
anything. The structure diagram is an M1 view, so it draws exactly one
`motors` node; there is nothing in the model (and so nothing in the
diagram) for an individual motor to be.

In [ ]:
motors = model.find("Drone::QuadCopter::motors")
print(f"{motors.qualified_name} : {motors.types[0]} [{motors.multiplicity.upper}]")

# one element, multiplicity four: the "4" is data on the description
assert sum(1 for e in model.iter_tree() if e.name == "motors") == 1
assert motors.multiplicity.upper.value == 4

## M0: the interpretation has four, each with a name

`m0.interpret` builds the population the description denotes: four
`Motor` **individuals** and four `Propeller` individuals with stable
`qname#index` ids, plus the chassis, battery, and camera singletons.
Roll-ups run over the individuals that actually exist --
`sum(motors.mass)` adds four real masses, where the M1 `totalMass`
build-up hand-encodes `4.0 * 0.055 + 4.0 * 0.015`. Tutorial 9 is the
deep dive; here the population is the cast of characters for the 3D
scene.

In [ ]:
quad = m0.interpret(model, "Drone::QuadCopter")
for individual in quad.individuals():
    print(individual.id)

motor_ids = [motor.id for motor in quad.root.slots["motors"]]
prop_ids = [prop.id for prop in quad.root.slots["propellers"]]
assert motor_ids == [f"Drone::QuadCopter#0.motors#{i}" for i in range(4)]
assert prop_ids == [f"Drone::QuadCopter#0.propellers#{i}" for i in range(4)]
assert quad.rollup("sum(motors.mass)") == 4 * 0.055  # over actual individuals
print("\nsum(motors.mass) over the population:", quad.rollup("sum(motors.mass)"), "kg")

## The 3D scene is a rendering of the M0 population

`drone_geometry` normally merges each part kind into one mesh (one
draw call each). `split_instances=True` keeps the motor and prop
instances separate -- `motor1` .. `motor4`, `prop1` .. `prop4`, the
same children the cadquery assembly exports -- so each can carry its
own identity key. The part map stamps **M0 individual ids** as those
keys: each motor can renders a motor individual, each prop disk a
propeller individual, the frame renders the chassis, the violet pod
the camera, and the ESC has no model part and stays untagged (inert).
Sizes and placements come from the model's own attribute values --
including the camera's mounting offsets and boresight, which the
geometric checks below will judge.

The rule that ties an individual key back to M1 is
`link.individual_qname`: strip each dotted segment's `#index` and join
with `::` -- so `Drone::QuadCopter#0.motors#2` *belongs to* the usage
`Drone::QuadCopter::motors`. That one derivation is the whole
M0 -> M1 projection.

In [ ]:
interp = longeron.Interpreter(model)
camera = dict(quad.root.slots["camera"].slots)  # x/y/z, azimuth/elevation/fov
mesh = geometry.drone_geometry(
    prop_diameter_in=interp.evaluate("Drone::Propeller::diameter") / geometry.IN,
    motor_mass=interp.evaluate("Drone::Motor::mass"),
    battery_mass=interp.evaluate("Drone::Battery::mass"),
    esc_mass=0.012,  # the 30.5 mm stack heuristic; no ESC in the model
    split_instances=True,
    camera=camera,
)
print([part["name"] for part in mesh["parts"]])

PART_MAP = {
    "frame": quad.root.slots["chassis"].id,
    "battery": quad.root.slots["battery"].id,
    "camera": quad.root.slots["camera"].id,
    **{f"motor{i + 1}": motor_id for i, motor_id in enumerate(motor_ids)},
    **{f"prop{i + 1}": prop_id for i, prop_id in enumerate(prop_ids)},
}

assert link.individual_qname("Drone::QuadCopter#0.motors#2") == "Drone::QuadCopter::motors"
assert link.individual_qname("Drone::QuadCopter::motors") is None  # not an individual id

## Side by side, cross-linked

`link_selection` stamps the keys onto the viewer's mesh and wires both
directions; it returns `unlink` for disposal. `on_pick` is the M0 tap:
it receives every raycaster report exactly as written -- the picked
individual id, or `[]` for a background click -- *before* the
selection is projected to M1, so nothing the diagram cannot represent
gets lost.

**Try it in JupyterLab:** click `motors` in the diagram and watch all
four motor cans pop; then click a single motor can and watch the
diagram select the one `motors` node while `picked` records the
individual.

In [ ]:
HEIGHT = 650

structure = diagrams.structure_diagram(model, height=f"{HEIGHT}px")  # match the 3D viewer's span
viewer = viewer3d.mesh_viewer(
    mesh, label="Drone::QuadCopter -- one M0 interpretation", width_px=HEIGHT, height_px=HEIGHT
)

picked: list = []
unlink = link.link_selection(structure, viewer, model, part_map=PART_MAP, on_pick=picked.append)

# the viewer's stage is responsive: its rendered height is containerWidth /
# aspect, so a FIXED container width pins it at exactly height_px (480);
# the diagram takes the remaining width at the same fixed height
viewer.layout = W.Layout(width=f"{HEIGHT}px", flex="0 0 auto")
structure.layout.width = "auto"
structure.layout.flex = "1 1 auto"
combined = W.HBox(
    [structure, viewer], layout=W.Layout(align_items="stretch", width="100%", overflow="hidden")
)
combined

## M1 -> M0: the fan-out, made visible

Selecting the *one* `motors` node highlights *four* motor meshes: a
usage matches every key that derives from it. Selecting the `Motor`
definition reaches the same individuals through the usage it types,
and selecting the whole `QuadCopter` matches the entire rendered
population. The cells below drive the same traitlets a browser click
writes, so the claims hold headless.

In [ ]:
# a click on the one M1 usage -> all four M0 individuals light up
structure.view.selection.ids = ["Drone::QuadCopter::motors"]
assert json.loads(viewer.highlight_json) == sorted(motor_ids)

# the definition reaches the same population through the usage it types
structure.view.selection.ids = ["Drone::Propeller"]
assert json.loads(viewer.highlight_json) == sorted(prop_ids)

# the whole assembly -> chassis + battery + camera + all eight rotor parts
structure.view.selection.ids = ["Drone::QuadCopter"]
assert json.loads(viewer.highlight_json) == sorted(set(PART_MAP.values()))

# something not rendered clears rather than dims
structure.view.selection.ids = ["Drone::HoverTime"]
assert viewer.highlight_json == "[]"

## M0 -> M1: a projection that keeps the individual

The reverse direction is many-to-one: the diagram has no node for
`motors#2`, so picking the third motor can selects the one `motors`
usage -- and the forward direction immediately fans back out, so all
four instances stay lit. The pick itself is not lost: `on_pick` got
the individual id, ready for an M0-side reaction -- look the
individual up in the population, trace it, log it.

In [ ]:
# what the canvas raycaster writes when you click the third motor can
viewer.picked_json = json.dumps(["Drone::QuadCopter#0.motors#2"])
assert list(structure.view.selection.ids) == ["Drone::QuadCopter::motors"]  # M1
assert json.loads(viewer.highlight_json) == sorted(motor_ids)  # ... fans back out
assert picked[-1] == ["Drone::QuadCopter#0.motors#2"]  # M0: the individual, kept

individual = next(i for i in quad.individuals() if i.id == picked[-1][0])
print("picked:", individual, " mass:", individual.slots["mass"], "kg")

# a background click clears both panes and reports [] on the tap
viewer.picked_json = json.dumps([])
assert picked[-1] == []
assert list(structure.view.selection.ids) == []
assert viewer.highlight_json == "[]"

# leave the fan-out visible for a live front-end
structure.view.selection.ids = ["Drone::QuadCopter::motors"]
assert json.loads(viewer.highlight_json) == sorted(motor_ids)
print("M1 <-> M0 round trip: all assertions passed")

## Which plane to think in

Think at **M1** when the question is about the *design*: architecture
and trades quantify over usages and definitions -- swap the motor
definition, re-run the study, compare mixes (tutorials 6-7). Think at
**M0** when the question is about a *population*: roll-ups that weigh
what actually exists, per-individual identities and traces,
Monte-Carlo over drawn configurations (tutorial 9). The linked panes
above are the two planes side by side: the diagram can only ever
select descriptions, the scene renders one interpretation of them, and
`individual_qname` is the entire bridge.

Headless caveat: everything above ran without a browser because both
front-ends are pure painters over synced traitlets. The pixels -- the
emissive pop, the dimming, a real raycast from a canvas click -- need
JupyterLab (`pixi run lab`).

## Geometric requirements: the model asks, the kernel measures

`examples/drone.sysml` carries an `installation` requirement group
with two *geometric* requirements, each with a `measure`, a utility
shape, and a weight -- so they land on tutorial 13's scoreboard:

- **`clearView`** -- the camera's view cone shall contain no part of
  the drone's own airframe (`occludedFraction <= 0.0`);
- **`propClearance`** -- no propeller disc shall overlap any other
  component (`discOverlapVolume <= 0.0`).

The measures are computed **kernel-side**, and they are CAD-native:
`geometry.camera_occlusion` builds a view CONE solid at the camera
(apex at the lens, axis along the boresight, half-angle
`fieldOfView / 2`, reaching one airframe bounding-box diagonal) and
boolean-intersects it with every other component's parametric solid --
the same solids `geometry.to_cadquery` builds -- reporting intersected
volume over cone volume; `geometry.disc_overlap` intersects each
propeller disc (a thin cylinder solid) with every other component and
reports the total overlap volume. With the `cad` extra installed the
booleans are exact (the OCC kernel); without it the same measures are
estimated by a deterministic volume quadrature over the mesh
triangles -- a clean installation reads exactly 0.0 either way, and
every report says which `engine` produced it. The model declares the
measure names as unvalued attributes; the wiring is one call --
`scoreboard(model, values=geometry.geometry_checks(mesh))` injects the
readings as evaluation-frame bindings. Nothing is baked into the model
file, and an unmeasured scoreboard stays honestly hatched.

In [ ]:
from longeron.analysis.scoreboard import scoreboard

checks = geometry.geometry_checks(mesh)
print("measures from the geometry:", checks)

board = scoreboard(model, values=checks)
print(board)

# the stock design point satisfies both geometric requirements exactly:
# nothing pokes into the view cone, and no disc touches anything
assert checks["occludedFraction"] <= 0.0  # clearView
assert checks["discOverlapVolume"] <= 0.0  # propClearance
rows = {row.name: row for row in board.table()}
assert rows["clearView"].utility > 0.9
board.widget()

### A violating variant, painted where it hurts

What-ifs need no model edits: `camera_occlusion` takes an explicit
camera mapping, so yaw the same camera 180 degrees -- looking straight
back through the airframe -- and the view cone impales the battery,
frame, and ESC. `occlusion_report` names the offending parts with
their intersection volumes, and the linked viewer's highlight seam
(the same one the diagram clicks drive) paints them in 3D. The
scoreboard goes red on `clearView`.

The prop swap fails the other requirement: a frame does not grow when
a bigger prop is bolted on, so passing the stock `motor_spacing` with
12-inch props overlaps the discs -- `disc_overlap` reports the
lens-shaped volumes where neighbouring discs cut into each other.

In [ ]:
# camera yawed 180 degrees: the airframe pokes into the view cone
report = geometry.occlusion_report(mesh, camera={**camera, "azimuth": 180.0})
print(f"occludedFraction looking backward: {report['occludedFraction']:.4f}")
print("obstructions (m^3):", report["obstructions"])
assert report["occludedFraction"] > 0.0  # clearView violated

bad_board = scoreboard(model, values={**checks, "occludedFraction": report["occludedFraction"]})
print(bad_board)

# paint the offenders in the linked 3D scene (the same highlight seam)
viewer.highlight([PART_MAP.get(name, name) for name in report["obstructions"]])

# the prop swap: 12" discs on the frame sized for 10" props overlap
stock_spacing = interp.evaluate("Drone::Propeller::diameter") + 0.02
oversized = geometry.drone_geometry(
    prop_diameter_in=12.0,
    motor_mass=interp.evaluate("Drone::Motor::mass"),
    battery_mass=interp.evaluate("Drone::Battery::mass"),
    esc_mass=0.012,
    split_instances=True,
    camera=camera,
    motor_spacing=stock_spacing,
)
for row in geometry.overlap_report(oversized):
    into = ", ".join(row["parts"]) or "nothing"
    print(f"  {row['disc']}: {row['overlap'] * 1e6:.1f} cm^3 of overlap, into {into}")
assert geometry.disc_overlap(oversized) > 0.0  # propClearance violated
bad_board.widget()

## The mission on a real globe

The same model also *behaves*: tutorial 4's `FlightStates` machine
executes, and `longeron.analysis.mission3d` maps a recorded execution
onto a waypoint route over a CesiumJS globe -- takeoff climbs, the
`flying` interval spends the route, `landing` descends, and the label
above the drone follows the ACTIVE STATE through the mission, so the
state machine is visibly driving the animation. And the drone is THIS
drone: pass the same to-scale mesh the linked scene above renders
(`mesh=`) and it exports to an in-house binary glTF that flies the
route nose-first over Esri satellite imagery (tokenless, the default
base -- `imagery='plain'` bares the globe, `'osm'` brings street
tiles). Scrub with Cesium's native timeline. (CesiumJS loads from a
CDN at view time; offline front-ends -- including the docs build --
degrade to a printed notice, exactly like the 3D viewer's three.js.)

And it flies like a *multirotor*: props level in the vertical
phases, and in cruise the **model's own physics** sets the forward
tilt -- `cruiseTilt = min(arccos(m g / T_continuous), 25 deg ops cap)`
(see the Attitude calc defs in `drone.sysml`) rides into the
animation as sampled quaternions. The same physics prices the route
in minutes, scored against the model's mission-time budget below.


In [ ]:
from longeron.analysis import mission3d

WAYPOINTS = [  # a small loop over Piedmont Park, midtown Atlanta (lat, lon, alt m MSL)
    (33.7813, -84.3833, 350.0),
    (33.7885, -84.3785, 390.0),
    (33.7900, -84.3695, 380.0),
    (33.7838, -84.3690, 360.0),
    (33.7770, -84.3825, 350.0),
]
tilt = mission3d.model_tilt(interp, "Drone::QuadCopter")  # the model's cruiseTilt calc
track = mission3d.from_replay(
    interp,
    "Drone::FlightStates",
    [2.0, "launch", 6.0, "airborne", 150.0, "low_battery", 10.0, "touchdown"],
    waypoints=WAYPOINTS,
    ground_alt=300.0,  # midtown Atlanta sits ~300 m MSL
    tilt_deg=tilt,  # cruise attitude FROM THE MODEL's physics
)
print(
    f"{track.name}: {track.duration:.0f} s of flight, {len(track.samples)} samples, "
    f"{tilt:.0f} deg cruise tilt"
)
mission3d.mission_viewer(track, mesh=mesh, height_px=420)

In [ ]:
# Mission time is an analysis: the kernel measures the route legs, the MODEL
# computes the physics (MissionTime at maxCruiseSpeed, which cruiseTilt sets)
values = mission3d.mission_values(interp, WAYPOINTS, ground_alt=300.0)
print(
    f"route {values['routeM'] / 1000:.1f} km at {values['cruiseSpeedMps']:.1f} m/s "
    f"-> {values['missionMinutes']:.1f} min against the 6.0 min budget"
)
board = scoreboard(model, values={**checks, **values})
assert {r.name: r for r in board.table()}["missionTime"].utility > 0.7  # green

# the one-line what-if: a 0.78 kg payload eats the continuous-thrust margin
# -- the tilt ceiling collapses, cruise slows, and the mission budget BUSTS
bust = mission3d.mission_values(interp, WAYPOINTS, ground_alt=300.0, payload_mass=0.78)
print(
    f"payload 0.78 kg: tilt {bust['cruiseTiltDeg']:.1f} deg, "
    f"{bust['cruiseSpeedMps']:.1f} m/s -> {bust['missionMinutes']:.1f} min: BUST"
)
bust_board = scoreboard(model, values={**checks, **bust})
assert {r.name: r for r in bust_board.table()}["missionTime"].utility == 0.0  # red
board.widget()